# 🏢 Web Scraping — Tipo de Inversión y UEI

**Fuente:** https://ofi5.mef.gob.pe/ssi/ (modal *Ver Resumen*)

Extrae por CUI: **Tipo de Inversión** y **Unidad Ejecutora de Inversiones (UEI)**

### 📋 Instrucciones
1. Coloca `CUI.xlsx` en la **misma carpeta** que este notebook (columna `CUI`)
2. Edita solo la celda de **Configuración** (PASO 2)
3. Ejecuta en orden con **Shift+Enter**

### ⚙️ PASO 1 — Instalación
> Solo la primera vez.

In [ ]:
!pip install selenium webdriver-manager openpyxl pandas

### ⚙️ PASO 2 — Configuración
> **⚠️ Única celda que debes editar.**

In [ ]:
from pathlib import Path

CARPETA_BASE = Path.cwd()

NOMBRE_ARCHIVO_ENTRADA = "CUI.xlsx"             # <- Cambia si tu archivo tiene otro nombre
NOMBRE_ARCHIVO_SALIDA  = "resultados_UEI.xlsx"  # <- Nombre del archivo de resultados

RUTA_ENTRADA = CARPETA_BASE / NOMBRE_ARCHIVO_ENTRADA
RUTA_SALIDA  = CARPETA_BASE / NOMBRE_ARCHIVO_SALIDA

MAX_REINTENTOS = 2
MODO_VISIBLE   = True
GUARDAR_CADA   = 5
TIMEOUT_PAGINA   = 20
TIMEOUT_ELEMENTO = 10
TIMEOUT_MODAL    = 10

print(f"📁 Carpeta: {CARPETA_BASE}")
print(f"{'✅' if RUTA_ENTRADA.exists() else '❌ NO ENCONTRADO'} {NOMBRE_ARCHIVO_ENTRADA}")

### ⚙️ PASO 3 — Importar librerías

In [ ]:
import pandas as pd
import time
from selenium import webdriver
from selenium.webdriver import Chrome
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait as Wait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
print("✅ Librerías listas")

### ⚙️ PASO 4 — Funciones de checkpoint y navegador

In [ ]:
def cargar_progreso():
    if RUTA_SALIDA.exists():
        try:
            df = pd.read_excel(RUTA_SALIDA)
            print(f"📥 Progreso: {len(df)} CUIs ya procesados")
            return df.to_dict('records')
        except Exception as e:
            print(f"⚠️ Error leyendo progreso: {e}")
    return []

def obtener_pendientes(completa, procesados):
    if not procesados:
        print(f"📋 Todos pendientes: {len(completa)}")
        return completa
    cuis_ok = {str(r['CUI']) for r in procesados}
    pendientes = [cui for cui in completa if str(cui) not in cuis_ok]
    if pendientes:
        print(f"⏳ Pendientes: {len(pendientes)}")
    return pendientes

def guardar(resultados):
    try:
        pd.DataFrame(resultados).to_excel(RUTA_SALIDA, index=False)
    except Exception as e:
        print(f"⚠️ Error guardando: {e}")

def crear_driver():
    service = Service(ChromeDriverManager().install())
    options = webdriver.ChromeOptions()
    if not MODO_VISIBLE:
        options.add_argument("--headless=new")
    else:
        options.add_argument("--start-maximized")
    for arg in ["--disable-gpu","--no-sandbox","--disable-dev-shm-usage",
                "--disable-extensions","--disable-blink-features=AutomationControlled",
                "--disable-logging","--log-level=3","--silent"]:
        options.add_argument(arg)
    prefs = {"profile.managed_default_content_settings.images": 2,
             "profile.default_content_setting_values.notifications": 2,
             "profile.default_content_setting_values.geolocation": 2}
    options.add_experimental_option("prefs", prefs)
    options.add_experimental_option('excludeSwitches', ['enable-logging'])
    options.page_load_strategy = 'normal'
    driver = Chrome(service=service, options=options)
    driver.set_page_load_timeout(TIMEOUT_PAGINA)
    driver.set_script_timeout(15)
    return driver

print("✅ Funciones listas")

### ⚙️ PASO 5 — Funciones de scraping (modal SSI — Tipo de Inversión y UEI)

In [ ]:
def cerrar_ventanas_emergentes(driver):
    """Cierra modales Bootstrap que puedan estar abiertos."""
    try:
        driver.execute_script("""
            document.querySelectorAll('.modal.show').forEach(m => {
                m.style.display = 'none';
                m.classList.remove('show');
            });
            document.querySelectorAll('.modal-backdrop').forEach(b => b.remove());
            document.body.classList.remove('modal-open');
            document.body.style.overflow = '';
            document.body.style.paddingRight = '';
        """)
    except:
        pass

def esperar_modal_visible(driver, timeout=10):
    """Espera a que el modal 'Ver Resumen' esté visible con los datos de Tipo/UEI."""
    try:
        Wait(driver, timeout).until(
            EC.visibility_of_element_located((By.ID, "divResumenCont"))
        )
        time.sleep(0.3)
        tiene = driver.execute_script("""
            var modal = document.getElementById('divResumenCont');
            if (!modal || modal.children.length === 0 || modal.offsetParent === null) return false;
            return document.getElementById('td_tipinv_r') !== null &&
                   document.getElementById('td_uei_r') !== null;
        """)
        if not tiene:
            time.sleep(0.5)
            tiene = driver.execute_script(
                "return document.getElementById('td_tipinv_r') !== null;"
            )
        return tiene
    except TimeoutException:
        return False
    except Exception as e:
        print(f"⚠️ {str(e)[:20]}", end=" ")
        return False

def extraer_datos_modal(driver):
    """Extrae Tipo de Inversión y UEI del modal Ver Resumen."""
    try:
        datos = driver.execute_script("""
            function getText(id) {
                try {
                    var e = document.getElementById(id);
                    if (!e) return 'NO DISPONIBLE';
                    return (e.textContent || e.innerText || '').trim() || 'NO DISPONIBLE';
                } catch(err) { return 'NO DISPONIBLE'; }
            }
            return {
                tipo_inversion:   getText('td_tipinv_r'),
                unidad_ejecutora: getText('td_uei_r')
            };
        """)
        if all(v == 'NO DISPONIBLE' for v in datos.values()):
            print("⚠️ Sin datos", end=" ")
            return None
        return datos
    except Exception as e:
        print(f"⚠️ {str(e)[:20]}", end=" ")
        return None

def procesar_cui(driver, cui, intento=1):
    """Busca el CUI en el SSI, abre el modal y extrae Tipo de Inversión y UEI."""
    try:
        driver.get("https://ofi5.mef.gob.pe/ssi/")
        input_box = Wait(driver, TIMEOUT_ELEMENTO).until(
            EC.element_to_be_clickable((By.ID, "txt_cu"))
        )
        input_box.clear()
        input_box.send_keys(cui)
        Wait(driver, TIMEOUT_ELEMENTO).until(
            EC.element_to_be_clickable((By.CLASS_NAME, "btn_bus"))
        ).click()
        Wait(driver, TIMEOUT_ELEMENTO).until(
            EC.presence_of_element_located((By.ID, "td_cu"))
        )
        time.sleep(0.3)
        cerrar_ventanas_emergentes(driver)
        time.sleep(0.2)

        try:
            btn = Wait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "img[src*='resumen.png']"))
            )
            driver.execute_script("arguments[0].scrollIntoView(true);", btn)
            time.sleep(0.2)
            driver.execute_script("arguments[0].click();", btn)
        except:
            driver.execute_script("""
                var imgs = document.querySelectorAll('img[src*="resumen.png"]');
                if (imgs.length > 0) imgs[0].click();
            """)

        time.sleep(0.5)
        if not esperar_modal_visible(driver, TIMEOUT_MODAL):
            raise Exception("Modal no cargó")

        datos = extraer_datos_modal(driver)
        if not datos:
            raise Exception("Sin datos")

        try:
            driver.execute_script("""
                var btn = document.querySelector('button[data-bs-dismiss="modal"]');
                if (btn) btn.click();
            """)
            time.sleep(0.2)
        except:
            pass

        return datos

    except Exception as e:
        if intento < MAX_REINTENTOS:
            print(f"🔄{intento+1}...", end=" ")
            time.sleep(1)
            cerrar_ventanas_emergentes(driver)
            return procesar_cui(driver, cui, intento + 1)
        else:
            raise e

print("✅ Funciones de scraping SSI listas")

### ▶️ PASO 6 — Cargar CUIs e inicializar

In [ ]:
print("📂 Cargando CUIs...")
df_cui = pd.read_excel(RUTA_ENTRADA)
lista_completa = df_cui['CUI'].astype(str).tolist()
print(f"📊 Total: {len(lista_completa)}")

resultados = cargar_progreso()
lista_cuis = obtener_pendientes(lista_completa, resultados)

if not lista_cuis:
    print("🎉 ¡Todos los CUIs ya fueron procesados!")
else:
    driver = crear_driver()
    print(f"⚡ Listo para procesar {len(lista_cuis)} CUIs")

### ▶️ PASO 7 — Ejecutar el scraping
> ⚠️ No cierres Chrome. Interrumpe con ⏹️; re-ejecuta PASOS 6 y 7 para continuar.

In [ ]:
tiempo_inicio = time.time()
exitosos = 0
fallos   = 0

print(f"{'='*60}\nInicio: {time.strftime('%H:%M:%S')}\n{'='*60}\n")

try:
    for idx, cui in enumerate(lista_cuis, 1):
        t0 = time.time()
        print(f"[{idx}/{len(lista_cuis)}] {cui}:", end=" ")

        try:
            datos = procesar_cui(driver, cui)
            resultados.append({
                "CUI": cui,
                "Tipo de Inversión": datos['tipo_inversion'],
                "Unidad Ejecutora de Inversiones (UEI)": datos['unidad_ejecutora']
            })
            exitosos += 1
            print(f"✅ ({time.time()-t0:.1f}s)")
        except Exception:
            resultados.append({
                "CUI": cui,
                "Tipo de Inversión": "NO DISPONIBLE",
                "Unidad Ejecutora de Inversiones (UEI)": "NO DISPONIBLE"
            })
            fallos += 1
            print(f"❌ ({time.time()-t0:.1f}s)")

        if (idx % GUARDAR_CADA == 0) or (idx == len(lista_cuis)):
            guardar(resultados)
            if idx % GUARDAR_CADA == 0:
                print(f"   💾", end="")

        if idx % 10 == 0:
            t_el = time.time() - tiempo_inicio
            eta  = (len(lista_cuis) - idx) * (t_el / idx)
            print(f"\n   📊 {exitosos}✅ {fallos}❌ | {t_el:.0f}s | ETA:{eta:.0f}s\n")

except KeyboardInterrupt:
    print("\n⚠️ INTERRUMPIDO")
    guardar(resultados)
    driver.quit()
    print("💾 Guardado. Re-ejecuta PASOS 6 y 7 para continuar.")
else:
    driver.quit()
    guardar(resultados)
    t_total = time.time() - tiempo_inicio
    print(f"\n{'='*60}\n🏁 COMPLETADO\n{'='*60}")
    print(f"📊 {len(lista_cuis)} procesados | ✅ {exitosos} | ❌ {fallos}")
    print(f"⏱️  {t_total:.1f}s | 💾 {RUTA_SALIDA}")

### 📊 PASO 8 — Estadísticas (opcional)

In [ ]:
if RUTA_SALIDA.exists():
    df_final = pd.read_excel(RUTA_SALIDA)
    total    = len(df_final)
    ok       = len(df_final[df_final['Tipo de Inversión'] != 'NO DISPONIBLE'])
    print(f"Total: {total} | ✅ Con datos: {ok} ({ok/total*100:.1f}%) | ❌ Sin datos: {total-ok}")
    df_final.head(5)
else:
    print("⚠️ Ejecuta el scraping primero.")